# Filled posts experiments: data checks (ticket 2166)

Read-only. Month contiguity per feature dataset, and when dormancy data starts.

## 0. Setup (SageMaker only)

Run the next two cells once, then restart the kernel. If `pip` fails because Python is older than 3.11 or there is no internet, stop and paste the error.

In [ ]:
%pip install polars==1.44.1 scikit-learn==1.7.1 matplotlib pyarrow

In [ ]:
!rm -rf repo repo-2166
!git clone --depth 1 --branch 2166-fp-exp-1 https://github.com/NMDSdevopsServiceAdm/DataEngineering.git repo-2166
!git -C repo-2166 rev-parse HEAD

## 1. Settings

In [ ]:
# Mode 1, experiment:         VARIANT = "<name>", RUN_LABEL = "checks", BASELINE_RUN_LABEL = "baseline"
# Mode 2, second-seed rescore: RUN_LABEL = f"{VARIANT}_seed43", BASELINE_RUN_LABEL = "baseline_seed43"
# Mode 3, second-seed baseline: VARIANT = "none", RUN_LABEL = "baseline_seed43", BASELINE_RUN_LABEL = None, FOLD_SEED = 43
VARIANT = "none"
RUN_LABEL = "checks"
BASELINE_RUN_LABEL = "baseline"  # folds and jitter sample are reused from this run

DATA_BUCKET = "sfc-2167-model-eval-datasets"  # read-only; confirm the name. Never push to this data branch
RESULTS_PREFIX = "s3://skillsforcare/emr-notebooks/filled-posts-eval/"  # persistent, not a branch bucket

CACHE_LOCATIONS = True
RUN_BRIDGE = False
FIRST_FY = None
FOLD_SEED = 42  # only used when BASELINE_RUN_LABEL is None
N_FOLDS = 5
JITTER_SAMPLE_SIZE = 10_000
SMALL_GROUP_LOCATIONS = 30

LOCAL_ROOT = None


In [ ]:
import hashlib
import io
import json
import platform
import subprocess
import sys
import time
from datetime import date, datetime, timezone
from pathlib import Path

import boto3
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import sklearn

REPO_DIR = Path("repo-2166").resolve()
sys.path.insert(0, str(REPO_DIR))

from polars_utils import utils as polars_utils
from polars_utils.expressions import is_care_home, is_not_care_home
from projects._03_independent_cqc._01_filled_posts._04_model.registry.model_registry import (
    model_registry,
)
from projects._03_independent_cqc._01_filled_posts._04_model.utils import (
    cross_validation_utils as cv,
)
from projects._03_independent_cqc._01_filled_posts._04_model.utils import (
    model_utils,
    paths,
    versioning,
)
from projects._03_independent_cqc._01_filled_posts._05_workplace_estimates.fargate.utils.models import (
    non_res_with_and_without_dormancy_combined as combine,
)
from projects._03_independent_cqc._01_filled_posts._05_workplace_estimates.fargate.utils.models import (
    utils as model_join,
)
from projects._03_independent_cqc.utils import model_evaluation_utils as evaluation
from utils.column_names.ind_cqc_pipeline_columns import IndCqcColumns as IndCQC
from utils.column_names.ind_cqc_pipeline_columns import ModelEvaluationColumns as Eval
from utils.column_names.ind_cqc_pipeline_columns import ModelRegistryKeys as MRKeys
from utils.column_values.categorical_column_values import AscwdsFilteringRule

pl.Config.set_tbl_rows(60)
pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_width_chars(220)
pl.Config.set_fmt_str_lengths(60)

COMMIT = subprocess.run(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], capture_output=True, text=True
).stdout.strip()
try:
    import psutil

    memory = f"{psutil.virtual_memory().total / 2**30:.0f} GiB"
except ImportError:
    memory = "unknown (psutil not installed)"
print(f"python {platform.python_version()} on {platform.platform()}")
print(f"polars {pl.__version__}, scikit-learn {sklearn.__version__}, numpy {np.__version__}")
print(f"memory {memory}; repo commit {COMMIT or 'not found'}")
assert pl.__version__ == "1.44.1" and sklearn.__version__ == "1.7.1", (
    "install polars==1.44.1 and scikit-learn==1.7.1"
)

In [ ]:
KEYS = [IndCQC.location_id, IndCQC.cqc_location_import_date]
LOCATION = IndCQC.location_id
DATE = IndCQC.cqc_location_import_date
SERVICE = IndCQC.primary_service_type
KNOWN = IndCQC.ascwds_filled_posts_dedup_clean
PRED = IndCQC.prediction
FOLD = Eval.fold
BAND = Eval.size_band

CARE_HOME_MODEL = IndCQC.care_home_model
NON_RES_WITH = IndCQC.non_res_with_dormancy_model
NON_RES_WITHOUT = IndCQC.non_res_without_dormancy_model
NON_RES_COMBINED = IndCQC.non_res_combined_model
SERIES = [CARE_HOME_MODEL, NON_RES_WITH, NON_RES_WITHOUT, NON_RES_COMBINED]

DATASETS = [
    "01_filled_posts_02_cleaned",
    "01_filled_posts_03_imputed",
    *[
        f"01_filled_posts_04_features_{name}"
        for name in (CARE_HOME_MODEL, NON_RES_WITH, NON_RES_WITHOUT)
    ],
]


def dataset_location(dataset: str):
    if LOCAL_ROOT:
        return Path(LOCAL_ROOT) / "datasets" / f"dataset={dataset}"
    return f"s3://{DATA_BUCKET}/domain=03_ind_cqc/dataset={dataset}/"


def scan(dataset: str) -> pl.LazyFrame:
    return polars_utils.scan_parquet(dataset_location(dataset))


def source_files(dataset: str) -> list[dict]:
    # Used to spot a dataset changing between the baseline and a later run.
    if LOCAL_ROOT:
        return [
            {"key": f.name, "size": f.stat().st_size, "last_modified": f.stat().st_mtime}
            for f in sorted(dataset_location(dataset).glob("*.parquet"))
        ]
    pages = (
        boto3.client("s3")
        .get_paginator("list_objects_v2")
        .paginate(Bucket=DATA_BUCKET, Prefix=f"domain=03_ind_cqc/dataset={dataset}/")
    )
    return [
        {"key": o["Key"], "size": o["Size"], "last_modified": o["LastModified"].isoformat()}
        for page in pages
        for o in page.get("Contents", [])
    ]


def write_result(name: str, content: bytes, run_label: str = RUN_LABEL) -> None:
    if LOCAL_ROOT:
        path = Path(LOCAL_ROOT) / "results" / run_label / name
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(content)
        return
    bucket, key = RESULTS_PREFIX.removeprefix("s3://").split("/", 1)
    boto3.client("s3").put_object(Bucket=bucket, Key=f"{key}{run_label}/{name}", Body=content)


def read_result(run_label: str, name: str) -> bytes:
    if LOCAL_ROOT:
        return (Path(LOCAL_ROOT) / "results" / run_label / name).read_bytes()
    bucket, key = RESULTS_PREFIX.removeprefix("s3://").split("/", 1)
    return boto3.client("s3").get_object(Bucket=bucket, Key=f"{key}{run_label}/{name}")["Body"].read()


def write_parquet_result(name: str, df: pl.DataFrame) -> None:
    buffer = io.BytesIO()
    df.write_parquet(buffer)
    write_result(name, buffer.getvalue())


def read_parquet_result(run_label: str, name: str) -> pl.DataFrame:
    return pl.read_parquet(io.BytesIO(read_result(run_label, name)))


# Self-check: one read and one write, so a permissions problem shows up now.
print("test read, cleaned rows:", scan(DATASETS[0]).select(pl.len()).collect().item())
write_result("_write_test.txt", b"ok")
print("test write ok")

## Months contiguous?

The date index is a dense rank of import date. If dates are consecutive months, index = months elapsed minus a constant.

In [ ]:
FEATURES = {name: scan(f"01_filled_posts_04_features_{name}") for name in (CARE_HOME_MODEL, NON_RES_WITH)}
INDEXED = IndCQC.cqc_location_import_date_indexed
for name, lf in FEATURES.items():
    pairs_df = lf.select(DATE, INDEXED).unique().sort(DATE).collect()
    month_number = pairs_df[DATE].dt.year() * 12 + pairs_df[DATE].dt.month()
    gaps = month_number.diff().drop_nulls()
    offsets = (pairs_df[INDEXED] - month_number).n_unique()
    print(f"{name}: {pairs_df.height} dates, {pairs_df[DATE].min()} to {pairs_df[DATE].max()}")
    print(f"  duplicate months: {int((gaps == 0).sum())}, gaps (months skipped): {int((gaps > 1).sum())}, index minus month number takes {offsets} value(s)")
    print("  contiguous" if offsets == 1 and (gaps == 1).all() else "  NOT contiguous: months elapsed differs from the index")

## When dormancy data starts

With-dormancy features only hold rows with dormancy populated, so a location's first row is when its dormancy data starts. Locations starting in 2022, by the month they first appear.

In [ ]:
first_df = FEATURES[NON_RES_WITH].group_by(LOCATION).agg(pl.col(DATE).min().alias("first_date")).collect()
starts_df = first_df.group_by("first_date").agg(pl.len().alias("locations")).sort("first_date")
in_2022 = starts_df.filter(pl.col("first_date").dt.year() == 2022)
print(in_2022)
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
axes[0].bar([d.strftime("%Y-%m") for d in in_2022["first_date"]], in_2022["locations"].to_list())
axes[0].set_title("Locations by month their dormancy data starts (2022)")
axes[0].tick_params(axis="x", rotation=45)
axes[1].plot(starts_df["first_date"], starts_df["locations"])
axes[1].set_title("All months")
for ax in axes:
    ax.grid(True)
plt.tight_layout()
plt.show()
print(f"{in_2022['locations'].sum():,} of {first_df.height:,} locations start in 2022")